# Semantic Search and RAG
- After the publication of this paper: https://arxiv.org/abs/1810.04805, Google announced it was using it to power Google search: https://blog.google/products-and-platforms/products/search/search-language-understanding-bert/

- Bing also stated that: Starting from April of this year, we used large transformer models to deliver the largest quality improvements to our Bing customers in the past year. Link: https://azure.microsoft.com/en-us/blog/bing-delivers-its-largest-improvement-in-search-experience-using-azure-gpus/




## Categories of Semantic search

- Dense Retrieval: it relies on the concept of the embeddings, and turn the search problem into retrieving the nearest neighbors of the search query (after both the query and documents are convererted to embeddings). Check the img1

- Reranking:  A reranking language model is one of these steps and is tasked with scoring the relevance of a subset of results against the query; the order of results is then changed based on these scores. Check img2

- RAG: Generative search is a subset of a broader type of category of systems better called RAG systems. These are text generation systems that incorporate search capabilities to reduce hallucinations, increase factuality, and/or ground the generation model on a specific dataset. Check img3

## Dense Retrieval
- Points that are close together mean that the text they represent is similar. 
- So in the img4, text 1 and text 2 are more similar to each other (because they are near each other) than text 3 (because it’s farther away). Check img4

- Should text 3 even be returned as a result? That’s a decision for you, the system designer. It’s sometimes desirable to have a max threshold of similarity score to filter out irrelevant results (in case the corpus has no relevant results for the query). check img5

- Are a query and its best result semantically similar? Not always. This is why language models need to be trained on question-answer pairs to become better at retrieval. chekc img5


### Dense Retrieval example

- We'll use Cohere to search the wikipedia page for the movie Interstellar:
  - Step1: Get the text we want to make searchable and apply some light processing to chunk it into sentences
  - Ste2: Embed each sentence
  - Step3: Build the search index
  - Step4: Search and see the results

In [1]:
pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-openai 1.1.13 requires langchain-core<2.0.0,>=1.2.29, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.



  Using cached langchain-0.2.5-py3-none-any.whl.metadata (7.0 kB)
  Using cached faiss_cpu-1.8.0-cp310-cp310-win_amd64.whl.metadata (3.8 kB)
  Using cached cohere-5.5.8-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_community-0.2.5-py3-none-any.whl.metadata (2.5 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl.metadata (10 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached fastavro-1.12.1-cp310-cp310-win_amd64.whl.metadata (5.7 kB)
  Using cached parameterized-0.9.0-py2.py3-none-any.whl.metadata (18 kB)
  Using cached types_requests-2.33.0.20260408-py3-none-any.whl.metadata (2.0 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata 

### Cohere Model
Link: 
  - https://dashboard.cohere.com/
  - https://docs.cohere.com/docs/sem-search-quickstart

In [2]:
import cohere
api_key = "YEQiiMe07mAo5A3BMjJA7cDMYJ1Szbuv9eAKvi8t"
co = cohere.Client(api_key)



In [5]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [26]:
# Embedding the Text Chunks
import numpy as np

# Get the embeddings
response = co.embed(
  texts=texts,
  input_type="search_document",
  model="embed-v4.0"
).embeddings

embeds = np.array(response)
print(embeds.shape)

(15, 1536)


Before we can search, we need to build a search index.
-  An index stores the embeddings and is optimized to quickly retrieve the nearest neighbors even if we have a very large number of points:

In [27]:
# Build the search index
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))

We can now search the dataset using any query we want. We simply embed the query and present its embedding to the index, which will retrieve the most similar sentence from the Wikipedia article:

In [32]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

def search(query, number_of_results=3):

  # 1. Get the query's embedding
  query_embed = co.embed(texts=[query], 
                model="embed-v4.0",
                input_type="search_query",).embeddings[0]

  # 2. Retrieve the nearest neighbors
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})

  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  return results

In [33]:
query = "how precise was the science"
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics,1.293341
1,"Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar",1.564170
2,"Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time",1.630263


- the first result has the least distance, and so the most similar to the query.
- Notice that this wouldn't have been possible if we were only doing keywords search because the top result did not include the same keywords in the query.


### BM25
- We can actually verify that by defining a keyword search function to compare the two.
- We’ll use the BM25 algorithm, which is one of the leading lexical search methods.

In [34]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token=token.strip(string.punctuation)
    
    if len(token)>0 and token not in _stop_words.ENGLISH_STOP_WORDS:
        tokenized_doc.append(token)
    
    return tokenized_doc


In [37]:
from tqdm import tqdm
tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 15/15 [00:00<?, ?it/s]


In [40]:

def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))






In [43]:
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	0.000	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine
	0.000	Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind


- Note that the first result does not really answer the question despite it sharing the word “science” with the query.

### Caveats of dense retrieval
- We need to be aware of the drawback of the dense retrieval:
  - Even if the text does not contain answer, Still get results and their distance. Check the following example
  - - In cases like this, one possible heuristic is to set a threshold level—a maximum distance for relevance, for example.

In [44]:
Query='What is the mass of the moon?'
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics,1.293341
1,"Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar",1.564170
2,"Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time",1.630263


- Caveat2: Another caveats of dense retrieval is when a user wants to find the exact matching for a given query.
    - That's the case that is perfect for keyword search
    - That’s one reason why hybrid search, which includes both semantic search and keyword search, is advised instead of relying solely on dense retrieval.
- Caveat3: Dense retrieval find it challenging to work properly in domains other the ones that they were trained on
- Caveat4: A case where each senetnce contained a piece of information, and we showed queries that specifically ask for that infomration. What about questions whose answers span multiple sentences
   - This highlights one of the important design parameters of dense retrieval systems: 
     - what is the best way to chunk long texts?
     -  why do we need to chunk them in the first place?

## Chunking long texts
- One limitation of Transformer language models is that they are limited in context sizes, meaning we cannot feed them very long texts that go above the number of words or tokens that the model supports. So how do we embed long texts?

### One Vector per document:
- Embed only representative part of the document and ignoring the rest of the text.
   - Embed only title, or only the begenning of the document
 - This is useful, but: it leaves a lot of information unindexed and therefore unsearchable.
 - It is good, if you are sure that the begenning of the doc capture all the relevant information
- Embed doc in chunks, embed those chunks, and then aggrgating those chunks into a single vector
  - A downside of this method is that the results in a highly compressed vector that loses a lot of the information in the doc

### Multiple Vector per document
- we chunk the document into smaller pieces, and embed those chunks.
- Our search index then becomes that of chunk embeddings, not entire document embeddings.
- The chunking approach is better because it has full coverage of the text and because the vectors tend to capture individual concepts inside the text
-  This leads to a more expressive search index

**Approaches**:
- Each sentence is a chunk. The issue here is this could be too granular and the vectors don’t capture enough of the context.
- Each paragraph is a chunk. This is great if the text is made up of short paragraphs. Otherwise, it may be that every 3–8 sentences is a chunk.
- Some chunks derive a lot of their meaning from the text around them. So we can incorporate some context via:
    - Adding the title of the document to the chunk.
    - Adding some of the text before and after them to the chunk. This way, the chunks can overlap so they  include some surrounding text that also appears in adjacent chunks.

### Nearest neighbor search versus vector databases
- Once the query is embedded, we need to find the nearest vectors to it from our text archive. Check img11
- The most straightforward way to find the nearest neighbors is to calculate the distances between the query and the archive. That can easily be done with NumPy and is a reasonable approach if you have thousands or tens of thousands of vectors in your archive.
- As you scale beyond to the millions of vectors, an optimized approach for retrieval is to rely on approximate nearest neighbor search libraries like Annoy or FAISS.
- These allow you to retrieve results from massive indexes in milliseconds and some of them can improve their performance by utilizing GPUs and scaling to clusters of machines to serve very large indices.

- Another class of vector retrieval systems are 
 - vector databases like Weaviate or Pinecone. 
 - A vector database allows you to add or delete vectors without having to rebuild the index. 
 - They also provide ways to filter your search or customize it in ways beyond merely vector distances.

### Fine-tuning embedding models for dense retrieval
- In the case of the retrieval, we need to optimize text embedding and not simply token embedding.
- The process for this fine-tuning is to get training data composed of queries and relevant results.



## Reranking
- Check img13
### Reranking Example
- A reranker takes in the search query and a number of search results, and returns the optimal ordering of these documents so the most relevant ones to the query are higher in ranking.
-  Cohere’s Rerank endpoint is a simple way to start using a first reranker: https://docs.cohere.com/reference/rerank

In [45]:
query = "How precise was the science"
results = co.rerank(
    query=query,
    documents=texts,
    top_n=3,
    return_documents=True,
)

In [46]:
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.13689686),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.047624897),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine'), index=1, relevance_score=0.039052043)]

In [48]:
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score, result.document.text)

0 0.13689686 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.047624897 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2 0.039052043 It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


This shows the reranker is much more confident about the first result, assigning it a relevance score of 0.13, while the other results are scored much lower in relevance.

- In this basic example, we passed our reranker all 15 of our documents.
- our index would have thousands or millions of entries, and we need to shortlist, say one hundred or one thousand results and then present those to the reranker. This shortlisting step is called the **first stage** of the search pipeline.

- The first-stage retriever can be keyword search, dense retrieval, or better yet—hybrid search that uses both of them.
- Let’s tweak our keyword search function so it retrieves a list of the top 10 results using keyword search, then use rerank to choose the top 3 results from those 10:

In [49]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

In [50]:
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	0.000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects
	0.000	In the United States, it was first released on film stock, expanding to venues using digital projectors
	0.000	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.032	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine
	0.031	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar


- We see that keyword search did not assign scores to the results that share some of the keywords.
- In the second set of results, the reranker elevates the last  result appropriately as the most relevant result for the query.
- This is a toy example that gives us a glimpse of the effect, but in practice, such a pipeline significantly improves search quality.

### Open source retrieval and re-ranking

Link: https://github.com/huggingface/sentence-transformers/blob/main/examples/sentence_transformer/applications/retrieve_rerank/retrieve_rerank_simple_wikipedia.ipynb

### How reranking models work
- One popular way of building LLM search rerankers is to present the query and each result to an LLM working as a cross-encoder. 
- This means that a query and possible result are presented to the model at the same time allowing the model to view both these texts before it assigns a relevance score, as we can see in img14
- Read More:
 - https://arxiv.org/abs/1910.14424
 - https://arxiv.org/abs/2010.06467




### Retrieval Evaluation Metrics
- Semantic search is evaluated using metrics from the Information Retrieval (IR) field. 
- Let’s discuss one of these popular metrics: mean average precision (MAP)
- Evaluating search systems needs three major components: 
  - A text archive, 
  - A set of queries, 
  - and relevance judgments indicating which documents are relevant for each query. Check img9 

  - Rad More abourt Evalutaion metrics: https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-in-information-retrieval-1.html


## Retrieval-Augmented Generation (RAG)
- The mass adoption of LLMs quickly led to people asking them questions and expecting factual answers. 
- While the models can answer some questions correctly, they also confidently answer lots of questions incorrectly.
- Paper: The leading method the industry turned to remedy this behavior is RAG, described in the paper “Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks” (2020): https://proceedings.neurips.cc/paper/2020/file/6b493230205f780e1bc26945df7481e5-Paper.pdf
- Check img23

### Grounded Generation with an LLM API
- Let’s look at how to add a grounded generation step after the search results to create our first RAG system.
-  For this example, we’ll use Cohere’s managed LLM, which builds on the search systems we’ve seen earlier in the chapter. 
-  We’ll use embedding search to retrieve the top documents, then we’ll pass those to the co.chat endpoint along with the questions to provide a grounded answer:

In [51]:
query= "income generated"
#1- Retrieval
# We'll use embedding search. But Ideally we'd do hybrid
results = search(query)

# 2- Grounded generation
doc_dict=[
    {'text': text} for text in results['texts']
]
response = co.chat(
   message=query,
   documents=doc_dict,
)

print(response.text)

Query:'income generated'
Nearest neighbors:
The film Interstellar generated $677 million worldwide, with subsequent re-releases bringing the total to $773 million.


In [52]:
response



NonStreamedChatResponse(text='The film Interstellar generated $677 million worldwide, with subsequent re-releases bringing the total to $773 million.', generation_id='d9521018-1d50-4bac-a007-036abe2f1434', citations=[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_2'], type='TEXT_CONTENT'), ChatCitation(start=32, end=54, text='$677 million worldwide', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=61, end=119, text='subsequent re-releases bringing the total to $773 million.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_2', 'text': 'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'}, {'id': 'doc_0', 'text': 'The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'}], is_search_required=None, search_queries=None, search_results=None, finish_reason='COMPLETE', tool_calls=None, ch

In [53]:
response.citations

[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_2'], type='TEXT_CONTENT'),
 ChatCitation(start=32, end=54, text='$677 million worldwide', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=61, end=119, text='subsequent re-releases bringing the total to $773 million.', document_ids=['doc_0'], type='TEXT_CONTENT')]

### Example: RAG with Local Models


In [54]:
from langchain import LlamaCpp

In [56]:
# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="../Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

llama_context: n_ctx_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


#### Load the embedding Model
Let’s now load an embedding language model. In this example, we will choose the BAAI/bge-small-en-v1.5 model. 

In [57]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
)

C:\Users\pc\AppData\Local\Temp\ipykernel_28652\3858863509.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Use

We can now use the embedding model to set up our Vector database


In [58]:
from langchain.vectorstores import FAISS

# Create a local vector database
db=FAISS.from_texts(texts, embedding_model)

RAG Prompt

In [61]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA

# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""

In [62]:
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# RAG pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(),
    chain_type_kwargs={"prompt": prompt},
    verbose=True
)

In [ ]:
rag.invoke('Income generated')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Income generated',
 'result': " The information provided does not directly relate to the income generated by Interstellar. However, it is known that successful films often generate significant box office revenue and can earn from DVD sales, streaming platforms, merchandise, and international releases. For precise figures on Interstellar's financial success, one would need additional data regarding its total gross revenue and expenditure."}

In [65]:
rag.invoke('What is the income generated by the movie interstellar')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is the income generated by the movie interstellar',
 'result': ' I\'m sorry, but the information provided doesn\'t include specific details about the income generated by the movie "Interstellar." The data given pertains to its premiere date, effects creation company, awards nominations and wins, as well as its production team. To answer your question accurately, additional financial data would be required.'}

In [66]:
rag.invoke('give the story of the movie interstellar')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'give the story of the movie interstellar',
 'result': ' "Interstellar," directed, co-written, and produced by Christopher Nolan, is a 2014 epic science fiction film that centers around the story of pioneering astronaut Cooper (played by Matthew McConaughey) who travels through a wormhole in search of a new home for humanity. With his team aboard the spacecraft Endurance, Cooper explores distant planets and encounters various challenges, including time dilation as they venture near a supermassive black hole called Gargantua. The journey is driven by the desperate need to find habitable worlds for Earth\'s dying population due to crop failure caused by dust storms. Interstellar has gained acclaim for its complex narrative, ambitious visual effects (for which it won Best Visual Effects at the 87th Academy Awards), and thought-provoking themes involving space exploration, love across dimensions, and human survival. It has since become recognized by many as one of the best scienc

## Advanced RAG technique

### Query rewriting
- LLM will struggle with the search queries that are too verbose.
- Example: We have an essay due tomorrow. We have to write about some animal. I love penguins. I could write about them. But I could also write about dolphins. Are they animals? Maybe. Let’s do dolphins. Where do they live for example?
- This should actually be rewritten into a query like: Where do dolphins live?
- Cohere’s API, for example, has a dedicated query-rewriting mode for co.chat.

### Multi-query RAG
- User Question: Compare the financial results of Nvidia in 2020 vs. 2023
- We may find one document that contains the results for both years, but more likely, we’re better off making two search queries:
 - Query 1: “Nvidia 2020 financial results”
 - Query 2: “Nvidia 2023 financial results”

### Multi-hop RAG
- A more advanced question may require a series of sequential queries. Take for example a question like:
- User Question: “Who are the largest car manufacturers in 2023? Do they each make EVs or not?”
- To answer this, the system must first search for:
    - Step 1, Query 1: “largest car manufacturers 2023”
- Then after it gets this information (the result being Toyota, Volkswagen, and Hyundai), it should ask follow-up questions:
 - Step 2, Query 1: “Toyota Motor Corporation electric vehicles”
 - Step 2, Query 2: “Volkswagen AG electric vehicles”
 - Step 2, Query 3: “Hyundai Motor Company electric vehicles”

 ### Query Routing
 - An additional enhancement is to give the model the ability to search multiple data sources.
 - for example, specify for the model that if it gets a question about HR, it should search the company’s HR information system (e.g., Notion) but if the question is about customer data, that it should search the customer relationship management (CRM) (e.g., Salesforce).

 ### Agentic RAG
 - This new nature of the LLM starts to become closer and closer to an agent that acts on the world.
 - for example, specify for the model that if it gets a question about HR, it should search the company’s HR information system (e.g., Notion) but if the question is about customer data, that it should search the customer relationship management (CRM) (e.g., Salesforce).
 
 - Not all LLMs will have the RAG capabilities mentioned here. At the time of writing, likely only the largest managed models may be able to attempt this behavior. Thankfully, Cohere’s Command R+ excels at these tasks and is available as an open-weights model as well.
- Links: https://cohere.com/blog/command-r-plus-microsoft-azure
- Links:https://huggingface.co/CohereLabs/c4ai-command-r-plus

## RAG Evaluation
- Good paper to read about this topic: https://arxiv.org/abs/2304.09848.
- Result will eb evaluated along four axes
  - Fluency: Whether the generated text is fluent and cohesive.
  - Perceived utility: Whether the generated answer is helpful and informative.
  - Citation recall: The proportion of generated statements about the external world that are fully supported by their citations.
  - Citation precision: The proportion of generated citations that support their associated statements.

- Ragas also scores some additional useful metrics like:
 - Faithfulness: Whether the answer is consistent with the provided context
 - Answer relevance: How relevant the answer is to the question